# 00_repo_setup

This notebook performs the initial setup and sanity checks for the surgical video compression project.

## Goals
- Define all project paths
- Verify dataset folders exist
- Define phase label mappings
- Inspect annotation files
- Inspect video files
- Create output folders for later notebooks

## Notes
This notebook should remain lightweight and should not perform expensive frame extraction or model training.

In [1]:
# ============================================================
# 1. Imports
# ============================================================
import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

from PIL import Image

In [2]:
# ============================================================
# 2. Project root and dataset paths
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "cholec80"

VIDEOS_DIR = DATA_ROOT / "videos"
PHASE_ANN_DIR = DATA_ROOT / "phase_annotations"
TOOL_ANN_DIR = DATA_ROOT / "tool_annotations"

CRF_DIRS = {
    "CRF18": DATA_ROOT / "videos_CRF18",
    "CRF23": DATA_ROOT / "videos_CRF23",
    "CRF28": DATA_ROOT / "videos_CRF28",
    "CRF35": DATA_ROOT / "videos_CRF35",
    "CRF51": DATA_ROOT / "videos_CRF51",
}

FRAMES_SAMPLED_DIR = DATA_ROOT / "frames_sampled"
SPLITS_DIR = DATA_ROOT / "splits"
OUTPUTS_DIR = DATA_ROOT / "outputs"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)

PROJECT_ROOT = /Users/niranjani/Desktop/video-compression-project
DATA_ROOT    = /Users/niranjani/Desktop/video-compression-project/cholec80


In [3]:
# ============================================================
# 3. Create generated-data folders if missing
# ============================================================
FRAMES_SAMPLED_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUTPUT_DIRS = {
    "resnet18": OUTPUTS_DIR / "resnet18",
    "resnet18_lstm": OUTPUTS_DIR / "resnet18_lstm",
    "resnet18_transformer": OUTPUTS_DIR / "resnet18_transformer",
    "r3d18": OUTPUTS_DIR / "r3d18",
    "must_like": OUTPUTS_DIR / "must_like",
}

for model_name, model_dir in MODEL_OUTPUT_DIRS.items():
    for sub in ["checkpoints", "predictions", "metrics", "plots", "gradcam"]:
        (model_dir / sub).mkdir(parents=True, exist_ok=True)

print("Created / verified generated-data folders.")

Created / verified generated-data folders.


In [4]:
# ============================================================
# 4. Phase mapping
# ============================================================
PHASE_MAP = {
    "Preparation": 0,
    "CalotTriangleDissection": 1,
    "ClippingCutting": 2,
    "GallbladderDissection": 3,
    "GallbladderPackaging": 4,
    "CleaningCoagulation": 5,
    "GallbladderRetraction": 6,
}

ID_TO_PHASE = {v: k for k, v in PHASE_MAP.items()}

print("Number of phase classes:", len(PHASE_MAP))
print(json.dumps(PHASE_MAP, indent=2))

Number of phase classes: 7
{
  "Preparation": 0,
  "CalotTriangleDissection": 1,
  "ClippingCutting": 2,
  "GallbladderDissection": 3,
  "GallbladderPackaging": 4,
  "CleaningCoagulation": 5,
  "GallbladderRetraction": 6
}


## Folder sanity checks

We verify that the expected dataset folders exist before moving on.

In [5]:
# ============================================================
# 5. Check required folders
# ============================================================
required_paths = {
    "videos": VIDEOS_DIR,
    "phase_annotations": PHASE_ANN_DIR,
    "tool_annotations": TOOL_ANN_DIR,
    "videos_CRF18": CRF_DIRS["CRF18"],
    "videos_CRF23": CRF_DIRS["CRF23"],
    "videos_CRF28": CRF_DIRS["CRF28"],
    "videos_CRF35": CRF_DIRS["CRF35"],
    "videos_CRF51": CRF_DIRS["CRF51"],
}

status_rows = []
for name, path in required_paths.items():
    status_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "is_dir": path.is_dir() if path.exists() else False,
    })

status_df = pd.DataFrame(status_rows)
status_df

,name,path,exists,is_dir
0,videos,/Users/niranjani/Desktop/video-compression-pro...,True,True
1,phase_annotations,/Users/niranjani/Desktop/video-compression-pro...,True,True
2,tool_annotations,/Users/niranjani/Desktop/video-compression-pro...,True,True
3,videos_CRF18,/Users/niranjani/Desktop/video-compression-pro...,True,True
4,videos_CRF23,/Users/niranjani/Desktop/video-compression-pro...,True,True
5,videos_CRF28,/Users/niranjani/Desktop/video-compression-pro...,True,True
6,videos_CRF35,/Users/niranjani/Desktop/video-compression-pro...,True,True
7,videos_CRF51,/Users/niranjani/Desktop/video-compression-pro...,True,True


In [6]:
# ============================================================
# 6. Basic file counts
# ============================================================
def count_files(folder, suffix=None):
    if not folder.exists():
        return 0
    if suffix is None:
        return sum(1 for _ in folder.iterdir())
    return sum(1 for p in folder.iterdir() if p.suffix.lower() == suffix.lower())

count_rows = [
    {"folder": "videos", "count": count_files(VIDEOS_DIR, ".mp4")},
    {"folder": "phase_annotations", "count": count_files(PHASE_ANN_DIR, ".txt")},
    {"folder": "tool_annotations", "count": count_files(TOOL_ANN_DIR, ".txt")},
]

for level, folder in CRF_DIRS.items():
    count_rows.append({"folder": level, "count": count_files(folder, ".mp4")})

count_df = pd.DataFrame(count_rows)
count_df

,folder,count
0,videos,80
1,phase_annotations,80
2,tool_annotations,80
3,CRF18,80
4,CRF23,80
5,CRF28,80
6,CRF35,80
7,CRF51,80


## Inspect file names

This helps confirm naming conventions such as `video01.mp4` and `video01-phase.txt`.

In [7]:
# ============================================================
# 7. Preview filenames
# ============================================================
def preview_filenames(folder, n=10):
    if not folder.exists():
        return []
    return sorted([p.name for p in folder.iterdir()])[:n]

print("Videos:")
print(preview_filenames(VIDEOS_DIR, n=10))

print("\nPhase annotation files:")
print(preview_filenames(PHASE_ANN_DIR, n=10))

print("\nTool annotation files:")
print(preview_filenames(TOOL_ANN_DIR, n=10))

for level, folder in CRF_DIRS.items():
    print(f"\n{level}:")
    print(preview_filenames(folder, n=5))

Videos:
['video01-timestamp.txt', 'video01.mp4', 'video02-timestamp.txt', 'video02.mp4', 'video03-timestamp.txt', 'video03.mp4', 'video04-timestamp.txt', 'video04.mp4', 'video05-timestamp.txt', 'video05.mp4']

Phase annotation files:
['video01-phase.txt', 'video02-phase.txt', 'video03-phase.txt', 'video04-phase.txt', 'video05-phase.txt', 'video06-phase.txt', 'video07-phase.txt', 'video08-phase.txt', 'video09-phase.txt', 'video10-phase.txt']

Tool annotation files:
['video01-tool.txt', 'video02-tool.txt', 'video03-tool.txt', 'video04-tool.txt', 'video05-tool.txt', 'video06-tool.txt', 'video07-tool.txt', 'video08-tool.txt', 'video09-tool.txt', 'video10-tool.txt']

CRF18:
['_metadata_snapshot', 'manifest_CRF18.csv', 'summary_CRF18.json', 'verification_CRF18.csv', 'video01.mp4']

CRF23:
['.DS_Store', '_metadata_snapshot', 'manifest_CRF23.csv', 'summary_CRF23.json', 'verification_CRF23.csv']

CRF28:
['.DS_Store', '_metadata_snapshot', 'manifest_CRF28.csv', 'summary_CRF28.json', 'verificatio

## Parse one phase annotation file

We inspect the structure and make sure the expected columns are present.

In [8]:
# ============================================================
# 8. Inspect one phase annotation file
# ============================================================
sample_phase_file = PHASE_ANN_DIR / "video01-phase.txt"

with open(sample_phase_file, "r") as f:
    for i, line in enumerate(f):
        print(repr(line.rstrip("\n")))
        if i >= 9:
            break

'Frame\tPhase'
'0\tPreparation'
'1\tPreparation'
'2\tPreparation'
'3\tPreparation'
'4\tPreparation'
'5\tPreparation'
'6\tPreparation'
'7\tPreparation'
'8\tPreparation'


In [9]:
# ============================================================
# 9. Phase annotation parser
# ============================================================
def parse_phase_file(phase_file: Path) -> pd.DataFrame:
    rows = []
    with open(phase_file, "r") as f:
        lines = f.readlines()

    if lines and ("Frame" in lines[0] or "Phase" in lines[0]):
        lines = lines[1:]

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue

        frame_idx = int(parts[0])
        phase_name = parts[1].strip()

        rows.append({
            "frame_idx": frame_idx,
            "phase_name": phase_name,
            "phase_id": PHASE_MAP.get(phase_name, -1),
        })

    return pd.DataFrame(rows)

phase_df = parse_phase_file(sample_phase_file)
phase_df.head(10)

,frame_idx,phase_name,phase_id
0,0,Preparation,0
1,1,Preparation,0
2,2,Preparation,0
3,3,Preparation,0
4,4,Preparation,0
5,5,Preparation,0
6,6,Preparation,0
7,7,Preparation,0
8,8,Preparation,0
9,9,Preparation,0


In [10]:
# ============================================================
# 10. Check for unknown phase labels
# ============================================================
unknown_rows = phase_df[phase_df["phase_id"] == -1]
print("Unknown-label rows:", len(unknown_rows))

if len(unknown_rows) > 0:
    display(unknown_rows.head())
else:
    print("All labels mapped successfully.")

Unknown-label rows: 0
All labels mapped successfully.


In [11]:
# ============================================================
# 11. Phase counts for sample file
# ============================================================
phase_df["phase_name"].value_counts().sort_index()

phase_name
CalotTriangleDissection    16300
CleaningCoagulation         1825
ClippingCutting             5350
GallbladderDissection      14575
GallbladderPackaging        2450
GallbladderRetraction       2301
Preparation                  525
Name: count, dtype: int64

## Parse all phase annotation files

This checks whether the whole annotation folder is consistent.

In [12]:
# ============================================================
# 12. Parse all phase annotation files
# ============================================================
all_phase_tables = []

for phase_file in sorted(PHASE_ANN_DIR.glob("*.txt")):
    video_name = phase_file.name.replace("-phase.txt", "")
    df = parse_phase_file(phase_file)
    df["video"] = video_name
    all_phase_tables.append(df)

all_phase_df = pd.concat(all_phase_tables, ignore_index=True)

print("Total annotation rows:", len(all_phase_df))
print("Unique videos:", all_phase_df["video"].nunique())
print("Unique labels:", sorted(all_phase_df["phase_name"].unique()))

Total annotation rows: 4612532
Unique videos: 80
Unique labels: ['CalotTriangleDissection', 'CleaningCoagulation', 'ClippingCutting', 'GallbladderDissection', 'GallbladderPackaging', 'GallbladderRetraction', 'Preparation']


In [13]:
# ============================================================
# 13. Overall label distribution
# ============================================================
all_phase_df["phase_name"].value_counts().sort_values(ascending=False)

phase_name
CalotTriangleDissection    1870651
GallbladderDissection      1460825
CleaningCoagulation         358303
ClippingCutting             352000
Preparation                 214301
GallbladderPackaging        190450
GallbladderRetraction       166002
Name: count, dtype: int64

In [14]:
# ============================================================
# 14. Per-video annotation counts
# ============================================================
video_counts_df = (
    all_phase_df.groupby("video")
    .size()
    .reset_index(name="num_annotation_rows")
    .sort_values("video")
    .reset_index(drop=True)
)

video_counts_df.head(20)

,video,num_annotation_rows
0,video01,43326
1,video02,70976
2,video03,145701
3,video04,38051
4,video05,58601
5,video06,53826
6,video07,113926
7,video08,37976
8,video09,67551
9,video10,43726


## Video file checks

We verify that the original and compressed video files exist and follow the same naming pattern.

In [15]:
# ============================================================
# 15. Compare original vs compressed filenames
# ============================================================
orig_videos = sorted([p.name for p in VIDEOS_DIR.glob("*.mp4")])

comparison = {"orig": set(orig_videos)}
for level, folder in CRF_DIRS.items():
    comparison[level] = set(p.name for p in folder.glob("*.mp4"))

for level in CRF_DIRS.keys():
    missing_from_level = sorted(comparison["orig"] - comparison[level])
    extra_in_level = sorted(comparison[level] - comparison["orig"])

    print(f"\n{level}")
    print("Missing compared to orig:", len(missing_from_level))
    print("Extra compared to orig:", len(extra_in_level))

    if len(missing_from_level) > 0:
        print("First few missing:", missing_from_level[:5])
    if len(extra_in_level) > 0:
        print("First few extra:", extra_in_level[:5])


CRF18
Missing compared to orig: 0
Extra compared to orig: 0

CRF23
Missing compared to orig: 0
Extra compared to orig: 0

CRF28
Missing compared to orig: 0
Extra compared to orig: 0

CRF35
Missing compared to orig: 0
Extra compared to orig: 0

CRF51
Missing compared to orig: 0
Extra compared to orig: 0


## Optional: inspect video metadata with ffprobe

This checks that ffmpeg/ffprobe can see the videos properly.

In [16]:
# ============================================================
# 16. ffprobe helper
# ============================================================
import subprocess

def ffprobe_video(video_path: Path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,duration",
        "-of", "default=noprint_wrappers=1",
        str(video_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip()

sample_video = VIDEOS_DIR / "video01.mp4"
stdout, stderr = ffprobe_video(sample_video)

print("STDOUT:")
print(stdout)
print("\nSTDERR:")
print(stderr if stderr else "(none)")

STDOUT:
width=854
height=480
r_frame_rate=25/1
avg_frame_rate=25/1
duration=1733.040000
nb_frames=43326

STDERR:
(none)


In [17]:
# ============================================================
# 17. Check one sample video from each compression level
# ============================================================
video_check_rows = []

for name, folder in [("orig", VIDEOS_DIR)] + list(CRF_DIRS.items()):
    video_path = folder / "video01.mp4"
    exists = video_path.exists()

    row = {
        "level": name,
        "path": str(video_path),
        "exists": exists,
    }

    if exists:
        stdout, stderr = ffprobe_video(video_path)
        row["ffprobe_ok"] = (stderr.strip() == "")
        row["ffprobe_output"] = stdout
    else:
        row["ffprobe_ok"] = False
        row["ffprobe_output"] = ""

    video_check_rows.append(row)

video_check_df = pd.DataFrame(video_check_rows)
video_check_df[["level", "exists", "ffprobe_ok", "path"]]

,level,exists,ffprobe_ok,path
0,orig,True,True,/Users/niranjani/Desktop/video-compression-pro...
1,CRF18,True,True,/Users/niranjani/Desktop/video-compression-pro...
2,CRF23,True,True,/Users/niranjani/Desktop/video-compression-pro...
3,CRF28,True,True,/Users/niranjani/Desktop/video-compression-pro...
4,CRF35,True,True,/Users/niranjani/Desktop/video-compression-pro...
5,CRF51,True,True,/Users/niranjani/Desktop/video-compression-pro...


## Save a small setup summary

This can be useful later for confirming what the repo looked like when the project started.

In [18]:
# ============================================================
# 18. Save setup summary
# ============================================================
setup_summary = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "num_phase_classes": len(PHASE_MAP),
    "phase_map": PHASE_MAP,
    "folder_counts": count_df.to_dict(orient="records"),
    "num_annotation_rows": int(len(all_phase_df)),
    "num_annotation_videos": int(all_phase_df["video"].nunique()),
}

summary_path = OUTPUTS_DIR / "setup_summary.json"

with open(summary_path, "w") as f:
    json.dump(setup_summary, f, indent=2)

print("Saved:", summary_path)

Saved: /Users/niranjani/Desktop/video-compression-project/cholec80/outputs/setup_summary.json


## Expected outcome of this notebook

By the end of this notebook, you should know:

- all required folders exist
- the file naming conventions are consistent
- phase annotations parse correctly
- video files are readable
- generated-data folders are ready for later notebooks

In [19]:
# ============================================================
# 19. Clean video filename lists
# ============================================================
orig_video_files = sorted([p.name for p in VIDEOS_DIR.glob("video*.mp4")])

crf_video_files = {
    level: sorted([p.name for p in folder.glob("video*.mp4")])
    for level, folder in CRF_DIRS.items()
}

print("First 10 original mp4 files:")
print(orig_video_files[:10])

for level, files in crf_video_files.items():
    print(f"{level}: {len(files)} mp4 files")

First 10 original mp4 files:
['video01.mp4', 'video02.mp4', 'video03.mp4', 'video04.mp4', 'video05.mp4', 'video06.mp4', 'video07.mp4', 'video08.mp4', 'video09.mp4', 'video10.mp4']
CRF18: 80 mp4 files
CRF23: 80 mp4 files
CRF28: 80 mp4 files
CRF35: 80 mp4 files
CRF51: 80 mp4 files
